# Install albumentations


In [ ]:
!pip install segmentation-models-pytorch albumentations --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 128.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 107.5 MB/s eta 0:00:00


# Mount Google Drive

In [ ]:
# prompt: google driveと連携

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Import Libraries

In [ ]:
import os
import numpy as np
from PIL import Image
from tqdm import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

# Config

In [ ]:
class Config:
    # データパス
    train_img_dir = '/content/drive/MyDrive/dataset_split/train/images'
    train_mask_dir = '/content/drive/MyDrive/dataset_split/train/masks'
    val_img_dir = '/content/drive/MyDrive/dataset_split/val/images'
    val_mask_dir = '/content/drive/MyDrive/dataset_split/val/masks'
    save_dir = '/content/drive/MyDrive/shioda-lab-unet'
    best_model_path = f'{save_dir}/best_model.pth'

    # ハイパーパラメータ
    target_size = (1024, 1248)
    encoder_name = 'resnet18'
    encoder_weights = 'imagenet'
    in_channels = 3
    classes = 1
    batch_size = 2
    num_epochs = 50
    lr = 1e-3
    seed = 42

# Define Dataset

In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, target_size=Config.target_size):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.image_names = sorted([f for f in os.listdir(image_dir) if f.endswith('.bmp')])
        self.target_size = target_size

        self.transform = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(p=0.2),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
            A.Resize(target_size[0], target_size[1]),
            A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
            ToTensorV2()
        ])

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.image_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name.replace('.bmp', '.png'))

        image = np.array(Image.open(img_path).convert('RGB'))
        mask = np.array(Image.open(mask_path).convert('L'))  # バイナリマスク前提
        mask = (mask > 0).astype(np.uint8)

        augmented = self.transform(image=image, mask=mask)
        return augmented['image'], augmented['mask'].float()

# Define loss function

In [ ]:
def dice_loss(pred, target, smooth=1.):
    pred_prob = torch.sigmoid(pred).squeeze(1)
    intersection = (pred_prob * target).sum(dim=(1, 2))
    union = pred_prob.sum(dim=(1, 2)) + target.sum(dim=(1, 2))
    dice = (2. * intersection + smooth) / (union + smooth)
    return 1 - dice.mean()

In [ ]:
train_ds = SegmentationDataset(Config.train_img_dir, Config.train_mask_dir)
val_ds = SegmentationDataset(Config.val_img_dir, Config.val_mask_dir)
train_loader = DataLoader(train_ds, batch_size=Config.batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=Config.batch_size)

# Train

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = smp.UnetPlusPlus(
    encoder_name=Config.encoder_name,
    encoder_weights=Config.encoder_weights,
    in_channels=Config.in_channels,
    classes=Config.classes,
).to(device)

criterion = dice_loss
optimizer = optim.Adam(model.parameters(), lr=Config.lr)

# --- 学習ループ ---
os.makedirs(Config.save_dir, exist_ok=True)
best_loss = float('inf')

for epoch in range(Config.num_epochs):
    model.train()
    running_loss = 0.0
    for images, masks in tqdm(train_loader):
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{Config.num_epochs}, Loss: {avg_loss:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), Config.best_model_path)
        print("✅ モデルを保存しました（新しい最良）")

print("✅ 学習完了！")

100%|██████████| 72/72 [01:19<00:00,  1.11s/it]


Epoch 1/50, Loss: 0.8462
✅ モデルを保存しました（新しい最良）


100%|██████████| 72/72 [00:16<00:00,  4.26it/s]


Epoch 2/50, Loss: 0.3432
✅ モデルを保存しました（新しい最良）


100%|██████████| 72/72 [00:16<00:00,  4.24it/s]


...

Epoch 48/50, Loss: 0.1887


100%|██████████| 72/72 [00:15<00:00,  4.54it/s]


Epoch 49/50, Loss: 0.1823
✅ モデルを保存しました（新しい最良）


100%|██████████| 72/72 [00:17<00:00,  4.21it/s]

Epoch 50/50, Loss: 0.1771
✅ モデルを保存しました（新しい最良）
✅ 学習完了！
